In [1]:
import os

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

%load_ext autoreload
%autoreload 2
from breeding_blanket_3d import get_blanket_base_case

from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax_sn
from jax_sn.operators.transport_operator import TransportOperatorBatchedPallas
import jax_sbgeom as jsb

In [2]:
discrete_blanket, flux_surface,  data, nt_data, d_physical = get_blanket_base_case()

(4, 7, 7)


In [3]:
jax.devices()[0]

CudaDevice(id=0)

In [4]:
mesh_base = discrete_blanket.volume_mesh(flux_surface)
element = jax_sn.solution_domain.BasixLagrangianSimplex(1,3)
cross_sections = jax_sn.domain.cross_sections.CrossSectionsIndexed(discrete_blanket.layer_array, data)
domain = jax_sn.domain.Domain.from_mesh_and_cross_sections(mesh_base, element.face_template, cross_sections)
quadrature_set = jax_sn.quadrature_set.create_tn_quadrature_set(3)
solution_domain = jax_sn.solution_domain.SolutionDomain.from_element_and_domain(element, domain)

In [5]:
from breeding_blanket_3d import get_source

source = jnp.stack([get_source(discrete_blanket)] * solution_domain.n_basis, axis=-1)

In [6]:
mgop = jax_sn.operators.multi_group_operator.multi_group_operator_creation.create_multi_group_operator(solution_domain, quadrature_set,
                                                                                                 single_angle_transport_operator_cls=TransportOperatorBatchedPallas,
                                                                                                 single_angle_transport_operator_cls_kwargs={"batch_size" : 100})

E0401 19:01:28.724256   78756 slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %reduce_sum.9 = s64[] reduce(%constant.207, %constant.128), dimensions={0}, to_apply=%region_7.21, metadata={op_name="jit(_topologicalsorted_angle_from_domain_quadrature_set_scan)/while/body/closed_call/jit(compute_levels_domain_safe_jax)/jit(compute_levels_iterative_jax)/reduce_sum" source_file="/home/tbogaarts/jax-sn/src/jax_sn/tools/graph_tools.py" source_line=172 source_end_line=172 source_column=29 source_end_column=58}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
E0401 19:01:28.930236   78653 slow_operation_alarm.cc:140] Th

In [7]:
from warp_kernel_v2 import WarpSweep, LevelBasedParallelismSetup

lsetup = LevelBasedParallelismSetup.from_transport_operator(mgop.transport_operator)

Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.


In [8]:
lsetup.n_levels

931

In [9]:
lsetup.max_level_size(0)

np.int64(1094)

In [10]:
xapply = jnp.ones_like(mgop.transport_operator.out_structure())

result = jnp.zeros_like(xapply)

In [11]:
#result_orig = mgop.transport_operator.inv(xapply)

In [12]:
import warp as wp
wp.init()


Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080" (10 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/tbogaarts/.cache/warp/1.12.0


In [13]:
xs_wp = wp.from_jax(mgop.domain.cross_sections_indexed.map_to_full_array(0).total.block_until_ready())

In [14]:
warpsweep = WarpSweep.from_transport_operator(mgop.transport_operator, xs_wp)

In [17]:
result_2 = warpsweep.multi_angle_levels_prealloc(xs_wp, xapply, lsetup, result)

In [18]:
import timeit 
time = timeit.timeit(lambda: wp.to_jax(warpsweep.multi_angle_levels_prealloc(xs_wp, xapply, lsetup, result)).block_until_ready(), number=5)

print("Time Warp", time / 5)

Time Warp 0.18199701179983094


In [19]:
time_jax = timeit.timeit(lambda: mgop.transport_operator.inv(xapply).block_until_ready(), number=5)
print("Time JAX", time_jax / 5)

Time JAX 1.882318638599827


In [20]:
result_final = wp.to_jax(warpsweep.multi_angle_levels_prealloc(xs_wp, xapply, lsetup, result))

In [22]:
result_orig = mgop.transport_operator.inv(xapply)

In [23]:
import numpy as onp
onp.testing.assert_allclose(result_final, result_orig, rtol=1e-7, atol=1e-12)

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=1e-12

Mismatched elements: 541 / 160574400 (0.000337%)
Max absolute difference among violations: 9.615919e-12
Max relative difference among violations: 6.00477645
 ACTUAL: array([[[1.908007e+04, 1.854869e+04, 1.553296e+04, 1.488509e+04],
        [1.866237e+04, 1.533649e+04, 1.469175e+04, 1.412024e+04],
        [1.868871e+04, 1.532022e+04, 1.469877e+04, 1.411050e+04],...
 DESIRED: array([[[1.908007e+04, 1.854869e+04, 1.553296e+04, 1.488509e+04],
        [1.866237e+04, 1.533649e+04, 1.469175e+04, 1.412024e+04],
        [1.868871e+04, 1.532022e+04, 1.469877e+04, 1.411050e+04],...